# 07 — Hierarchical forecasting and reconciliation

A healthcare network consumes forecasts at three levels at once: clinics
roster against clinic-level numbers, regional managers budget against
regional totals, and the network plans capacity against the grand total. If
those numbers come from independent forecasts they will not add up — and the
first time a regional budget meeting and a clinic roster quote different
totals, trust in the whole system erodes.

**Reconciliation** produces one coherent set of numbers: clinic forecasts that
sum exactly to regional totals, which sum exactly to the network total. This
notebook implements and compares three transparent methods from
`clinic_forecast.reconciliation`:

| Method | Forecast level | Disaggregation | Typical strength |
| --- | --- | --- | --- |
| Bottom-up | Clinic | none (sum upward) | Best when the clinic model is strong |
| Top-down | Network | historical proportions | Robust totals; blind to clinic dynamics |
| Middle-out | Region | within-region proportions | Compromise when regions are stable |

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
pd.set_option("display.max_columns", 60)

from clinic_forecast.data import generate_network_data

data_path = PROJECT_ROOT / "data" / "processed" / "clinic_daily_usage.csv"
if data_path.exists():
    usage = pd.read_csv(data_path, parse_dates=["date"])
    metadata = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "clinic_metadata.csv")
else:
    network = generate_network_data()
    usage, metadata = network.usage, network.metadata
    usage["date"] = pd.to_datetime(usage["date"])

cutoff = usage["date"].max() - pd.Timedelta(days=28)
train, test = usage[usage["date"] <= cutoff], usage[usage["date"] > cutoff]

## The incoherence problem, demonstrated

Forecast the same 28 days twice: once by summing clinic-level ML forecasts,
once with a seasonal naive model on the network-total series. Both are
reasonable; they disagree — and neither finance nor operations should have to
arbitrate which number is "real".

In [2]:
from clinic_forecast.models.baseline import seasonal_naive_forecast
from clinic_forecast.models.global_ml import GlobalMLForecaster

model = GlobalMLForecaster().fit(train)
combined = pd.concat([train, test], ignore_index=True).sort_values(["clinic_id", "date"])
clinic_ml = model.predict_known_future(combined)
clinic_ml = clinic_ml[clinic_ml["date"] > cutoff][["clinic_id", "date", "forecast"]]

network_train = train.groupby("date", as_index=False)["visits"].sum()
network_train["clinic_id"] = "network"
network_test = test.groupby("date", as_index=False)["visits"].sum()
network_test["clinic_id"] = "network"
network_naive = seasonal_naive_forecast(train=network_train, future=network_test)

summed_ml = clinic_ml.groupby("date")["forecast"].sum()
direct_naive = network_naive.set_index("date")["forecast"]
gap = (summed_ml - direct_naive).abs().mean() / direct_naive.mean()
print(f"Mean disagreement between the two 'network totals': {gap:.1%}")

Mean disagreement between the two 'network totals': 27.5%


## Three reconciled hierarchies

Each method produces a full coherent hierarchy; `assert_coherent` verifies
the summation constraints hold to numerical tolerance at every level.

In [3]:
from clinic_forecast.reconciliation import (
    assert_coherent, build_hierarchy_frame,
    reconcile_bottom_up, reconcile_middle_out, reconcile_top_down,
)

region_train = (
    train.merge(metadata[["clinic_id", "region"]], on="clinic_id", how="left", suffixes=("", "_m"))
    .groupby(["region", "date"], as_index=False)["visits"].sum()
)
region_test = (
    test.merge(metadata[["clinic_id", "region"]], on="clinic_id", how="left", suffixes=("", "_m"))
    .groupby(["region", "date"], as_index=False)["visits"].sum()
)
region_naive = seasonal_naive_forecast(
    train=region_train, future=region_test, id_col="region"
)[["region", "date", "forecast"]]

hierarchies = {
    "bottom_up": reconcile_bottom_up(clinic_ml, metadata),
    "top_down": reconcile_top_down(
        network_naive[["date", "forecast"]], train, metadata
    ),
    "middle_out": reconcile_middle_out(region_naive, train, metadata),
}
for name, hierarchy in hierarchies.items():
    assert_coherent(hierarchy, metadata)
    print(f"{name:>10s}: coherent across clinic -> region -> network")

 bottom_up: coherent across clinic -> region -> network
  top_down: coherent across clinic -> region -> network
middle_out: coherent across clinic -> region -> network


## Accuracy at every level

Coherence is a constraint, not a goal in itself — the question is what each
method costs in accuracy at each level. Actuals are aggregated to the same
hierarchy and WAPE is computed per (method, level).

In [4]:
from clinic_forecast.evaluation import evaluate_forecasts

actual_hierarchy = build_hierarchy_frame(test, metadata, value_col="visits")

rows = []
for name, hierarchy in hierarchies.items():
    scored = hierarchy.merge(actual_hierarchy, on=["level", "node", "date"], how="inner")
    scored["model"] = name
    rows.append(scored)
scored_all = pd.concat(rows, ignore_index=True)

level_metrics = evaluate_forecasts(scored_all, group_cols=["level"])
level_metrics.pivot(index="model", columns="level", values="wape").round(1)

level,clinic,network,region
model,,,
bottom_up,21.3,7.9,13.4
middle_out,44.5,32.3,36.8
top_down,44.0,32.3,36.6


In [5]:
level_metrics.pivot(index="model", columns="level", values="bias").round(1)

level,clinic,network,region
model,,,
bottom_up,-2.9,-2.9,-2.9
middle_out,-1.1,-1.1,-1.1
top_down,-1.1,-1.1,-1.1


The classic pattern appears:

- **Bottom-up wins at clinic level** — it is the only method that actually
  models clinic dynamics; proportional splits cannot know that one clinic is
  surging while its neighbour declines.
- **The gap narrows at network level**, where idiosyncratic clinic errors
  partially cancel. Top-down's static proportions are its weakness: every
  clinic inherits the network's growth uniformly, which mis-serves clinics
  whose trends diverge (exactly what the changepoint structure in this data
  creates).
- **Middle-out sits between** — regional naive totals are decent, but
  within-region proportions still smear clinic-level dynamics.

**Recommendation for this network: bottom-up from the global ML model.** It
is coherent by construction, best at the level where staffing decisions are
made, and competitive at every aggregate level. Proportion-based methods earn
their place when bottom-level series are too short or too noisy to model —
e.g. newly opened clinics, which could be top-down allocated inside an
otherwise bottom-up hierarchy.

## Beyond this PoC

Optimal-combination reconciliation (MinT and relatives) uses the error
covariance structure to beat all single-direction methods, at the cost of
transparency and a specialised dependency. The simple methods here are the
right first step: they fix the coherence problem, are explainable in one
sentence each, and provide the baseline any fancier method must beat.